# Phase 9: Wave-Level Generation (WLG)
## The Universal Generation Engine — Thinking in Waves, Not Bytes

**Phase 8** proved FLUX can generate bytes. **Phase 9** replaces byte-by-byte generation with wave-by-wave generation — the same mechanism that will later drive image, audio, molecular, and sensor generation.

```
Phase 8:  Field → GRU → b → y → t → e → s  (serial, text-only, ~100 steps)
Phase 9:  Field → wave → wave → wave → wave  (parallel-decodable, universal, ~15 steps)
                    ↓       ↓       ↓       ↓
                  "The"  "future"  "of"   "AI"   (last-mile WaveToText)
```

**New components:** WaveChunker, WaveGenerator (universal core), WaveToText (text last-mile), ThermodynamicWaveSampler

### What this notebook does

1. Clone / pull FLUX repo from GitHub
2. Install dependencies + run `setup.py`
3. Initialize `PhaseLogger(phase=9)`, detect hardware, load `HF_TOKEN`
4. Load FLUX components (Phase 7 → Phase 8 fallback → fresh)
5. Build Phase 9 modules + load training data
6. Stage 1: WaveToText pre-training (~2-3 hours)
7. Stage 2: WaveGenerator training (~4-6 hours)
8. Save checkpoint + upload to HuggingFace Hub
9. Test 1: Wave Distribution Match
10. Test 2: Word Reconstruction Accuracy
11. Test 3: Full Pipeline English Words
12. Demo 1: Wave-Level Generation
13. Demo 2: Wave Sequence Visualization
14. Demo 3: Thermodynamic Sampling Trace
15. Interactive exploration
16. Results summary
17. View full log
18. Final upload (logs → HF Hub + GitHub)
19. Save artifacts to Kaggle output

### Secrets Required

- `HF_TOKEN`: HuggingFace API token (Kaggle → Add-ons → Secrets)

**Runtime:** ~4-6 hours total (Stage 1: ~15min, Stage 2: ~4-6hr)

---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import sys
import subprocess
import importlib
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

# ─────────────────────────────────────────────
# 1. Clone or Pull — always get the absolute
#    latest code from GitHub.
# ─────────────────────────────────────────────
if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")

    subprocess.run(
        ['git', 'remote', 'set-url', 'origin', REPO_URL],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )

    subprocess.run(['git', 'checkout', '.'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)
    subprocess.run(['git', 'clean', '-fd'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)

    fetch = subprocess.run(['git', 'fetch', '--all'], cwd=str(WORK_DIR),
                           capture_output=True, text=True)
    if fetch.returncode != 0:
        print(f"  ⚠ git fetch failed: {fetch.stderr.strip()}")

    # Show what will change before resetting
    diff = subprocess.run(
        ['git', 'diff', '--stat', 'HEAD', 'origin/main'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    if diff.stdout.strip():
        print(f"\n  Files changed since last pull:")
        for line in diff.stdout.strip().split('\n'):
            print(f"    {line}")
    else:
        print("  ℹ No file changes — already up to date")

    result = subprocess.run(
        ['git', 'reset', '--hard', 'origin/main'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(result.stdout.strip() or result.stderr.strip())

    head = subprocess.run(
        ['git', 'log', '--oneline', '-3'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(f"\n  Latest commits:\n{head.stdout.strip()}")
    print("\n✓ Pulled & reset to latest origin/main")
else:
    if WORK_DIR.exists():
        import shutil
        shutil.rmtree(str(WORK_DIR))
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")

# ─────────────────────────────────────────────
# 2. Flush cached Python modules so the kernel
#    picks up the freshly-pulled code.
# ─────────────────────────────────────────────
_stale = [m for m in sys.modules if any(
    m.startswith(p) for p in (
        'flux_large', 'flux_model', 'flux_generate', 'flux_trainer',
        'wave_chunker', 'wave_generator', 'wave_to_text', 'wave_sampler',
        'wave_decoder', 'train_wave_gen',
        'working_memory', 'episodic_memory', 'semantic_memory',
        'memory_router', 'consolidation', 'train_memory',
        'cgn', 'manifold', 'causal_graph', 'multi_timescale',
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'flux_utils', 'wave_types', 'interference',
        'attractor', 'field_ops', 'train_', 'demo_', 'test_',
        'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
    )
)]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"  ✓ Flushed {len(_stale)} cached modules: {_stale[:5]}{'...' if len(_stale) > 5 else ''}")

# ─────────────────────────────────────────────
# 3. Delete stale Phase 9 checkpoint so training
#    runs fresh with the updated code.
# ─────────────────────────────────────────────
_stale_ckpt = WORK_DIR / 'checkpoints' / 'phase9.phase.pt'
if _stale_ckpt.exists():
    _stale_ckpt.unlink()
    print(f"  ✓ Deleted stale checkpoint: {_stale_ckpt.name}")

subprocess.run(['python', 'setup.py'], cwd=str(WORK_DIR),
               capture_output=True, text=True)

# Quick sanity check: verify Phase 9 modules are present
_wg_path = WORK_DIR / 'phases' / 'phase9' / 'wave_generator.py'
_wg_text = _wg_path.read_text()
assert 'WaveGenerator' in _wg_text, "ERROR: wave_generator.py missing WaveGenerator!"
print(f"  ✓ wave_generator.py verified: WaveGenerator present")

_wtt_path = WORK_DIR / 'phases' / 'phase9' / 'wave_to_text.py'
_wtt_text = _wtt_path.read_text()
assert 'WaveToText' in _wtt_text, "ERROR: wave_to_text.py missing WaveToText!"
print(f"  ✓ wave_to_text.py verified: WaveToText present")

_wc_path = WORK_DIR / 'phases' / 'phase9' / 'wave_chunker.py'
_wc_text = _wc_path.read_text()
assert 'WaveChunker' in _wc_text, "ERROR: wave_chunker.py missing WaveChunker!"
print(f"  ✓ wave_chunker.py verified: WaveChunker present")

_tw_path = WORK_DIR / 'phases' / 'phase9' / 'train_wave_gen.py'
_tw_text = _tw_path.read_text()
assert 'Phase9Trainer' in _tw_text, "ERROR: train_wave_gen.py missing Phase9Trainer!"
print(f"  ✓ train_wave_gen.py verified: Phase9Trainer present")

print("✓ setup.py refreshed")
print(f"\nPhase 9 files:")
for f in sorted((WORK_DIR / 'phases' / 'phase9').iterdir()):
    if f.is_file() and f.suffix == '.py':
        print(f"  {f.name}")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q torch numpy scipy matplotlib tqdm rich transformers huggingface_hub datasets faiss-cpu
!python setup.py

print("  ✓ Dependencies installed")

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys, torch, platform, importlib
from pathlib import Path

# ── Add project paths ──
sys.path.insert(0, str(Path.cwd()))
for phase in ['phase1', 'phase2', 'phase3', 'phase4', 'phase5',
              'phase6', 'phase7', 'phase8', 'phase9']:
    sys.path.insert(0, str(Path.cwd() / 'phases' / phase))

# ── Force-reload all project modules ──
for _mod_name in list(sys.modules.keys()):
    if any(_mod_name.startswith(p) for p in (
        'flux_utils', 'flux_large', 'flux_model', 'flux_generate', 'flux_trainer',
        'wave_chunker', 'wave_generator', 'wave_to_text', 'wave_sampler',
        'wave_decoder', 'train_wave_gen',
        'working_memory', 'episodic_memory', 'semantic_memory',
        'memory_router', 'consolidation', 'train_memory',
        'cgn', 'manifold', 'causal_graph', 'multi_timescale',
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'wave_types', 'interference', 'attractor', 'field_ops',
        'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
    )):
        del sys.modules[_mod_name]

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint, verify_checkpoint_chain,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults, checkpoint_exists,
)

# ── Initialize Phase 9 Logger ──
log = PhaseLogger(phase=9)
log.separator("Phase 9: Wave-Level Generation")
log.cell_start("Cell 3 — Hardware & Secrets")

# ── Detect hardware ──
DEVICE = get_device()
hw = get_hardware_info()
log.info(f"Device: {DEVICE}")
log.info(f"PyTorch: {torch.__version__}")
log.info(f"Platform: {hw['platform']}")
for k, v in hw.items():
    print(f"  {k}: {v}")

if torch.cuda.is_available():
    log.info(f"GPU: {hw.get('gpu', 'N/A')}")
    log.info(f"VRAM: {hw.get('gpu_memory', 'N/A')}")
    log.info(f"CUDA: {hw.get('cuda', 'N/A')}")

# ── Load HuggingFace token ──
HF_TOKEN = _resolve_hf_token()
if HF_TOKEN:
    log.success("HuggingFace token loaded")
    print("  ✓ HF token loaded")
else:
    log.warning("No HuggingFace token — checkpoint upload will be skipped")
    print("  ⚠ No HF token — add HF_TOKEN to Kaggle Secrets for auto-upload")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

---
## Cell 4: Load FLUX Components (Phase 7 → Phase 8 fallback → Fresh)

Phase 9 is a **peer** to Phase 8, not a continuation. Both build on Phase 7 (CSE, field, GR, TL, CGN, memory, bridges).
- Phase 8 added WaveDecoder (byte-level generation)
- Phase 9 adds WaveGenerator (wave-level generation)

Phase 7 is the true dependency. Phase 8 is only a legacy fallback.

In [ ]:
log.cell_start("Cell 4 — Load FLUX Components")

import torch
import importlib

# Force-reimport phase modules
for _m in ['cse', 'field', 'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
           'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
           'cgn', 'manifold', 'causal_graph', 'multi_timescale',
           'working_memory', 'episodic_memory', 'semantic_memory',
           'memory_router', 'consolidation',
           'flux_large', 'flux_model', 'flux_generate', 'flux_trainer',
           'wave_decoder', 'wave_chunker', 'wave_generator', 'wave_to_text',
           'wave_sampler', 'train_wave_gen']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from flux_large import FLUXLarge
from flux_utils import checkpoint_exists, load_checkpoint

# Phase 9 is a PEER to Phase 8, not a continuation.
# Both build on Phase 7 (CSE, field, GR, TL, CGN, memory, bridges).
# Phase 8 added WaveDecoder (byte-level). Phase 9 adds WaveGenerator (wave-level).
# Phase 7 is the true dependency. Phase 8 is only a legacy fallback.

model = None

# 1. Try Phase 7 — the true foundation (has all Phases 1-7 trained)
if model is None:
    try:
        model = FLUXLarge.from_phase7_checkpoint(device=DEVICE)
        print("  ✓ Loaded Phase 7 (CSE, field, GR, TL, CGN, memory, bridges)")
        print("  ℹ Phase 9 builds wave-level generation on Phase 7 foundation")
    except Exception as e:
        print(f"  ℹ No Phase 7 checkpoint: {e}")

# 2. Legacy fallback — Phase 8 contains Phase 7 weights + WaveDecoder we don't use
if model is None:
    try:
        model = FLUXLarge.from_phase8_checkpoint(device=DEVICE)
        print("  ✓ Loaded Phase 8 as fallback (contains Phase 7 weights)")
        print("  ✗ WaveDecoder ignored — Phase 9 replaces it")
    except Exception as e:
        print(f"  ℹ No Phase 8 checkpoint: {e}")

# 3. Last resort — fresh init (untrained, poor quality)
if model is None:
    print("  ⚠ No checkpoints — using fresh FLUXLarge (untrained)")
    model = FLUXLarge(device=DEVICE)

# Freeze everything — Phase 9 only trains WaveGenerator + WaveToText
for param in model.parameters():
    param.requires_grad = False

# Quick smoke test
with torch.no_grad():
    wave = model.cse.encode("Hello world")
    wave_seq = wave.full.to(DEVICE)
    wave_vec = wave_seq.mean(dim=0)
    field_feats, sims, locs = model.field.query(wave_vec, k=4)
    merged = field_feats.mean(dim=0) + model.cgn(field_feats.mean(dim=0))

print(f"  ✓ Smoke test: CSE→{wave_seq.shape}, Field→{field_feats.shape}, Merged→{merged.shape}")

log.cell_end("Cell 4 — Load FLUX Components", "PASS")

---
## Cell 5: Build Phase 9 Modules + Load Training Data

Phase 9 introduces:
- **WaveChunker:** Segments text into wave-sized chunks
- **WaveGenerator:** Universal core — generates wave-by-wave (not byte-by-byte)
- **WaveToText:** Text last-mile decoder (wave → readable text)
- **ThermodynamicWaveSampler:** Energy-based sampling for wave generation

In [ ]:
log.cell_start("Cell 5 — Build Phase 9 Modules")

from train_wave_gen import (
    build_phase9_modules, Phase9Trainer, load_training_data,
    generate_text, PHASE9_CONFIG,
)
from wave_sampler import ThermodynamicWaveSampler

# Build fresh Phase 9 modules
chunker, generator, wtt = build_phase9_modules(device=DEVICE)

# Load training data — FLUX uses continuous stream learning (no epochs).
# We need enough documents for a single pass to reach max_steps.
# ~10 chunks/doc, batch_size=32 → need ~max_steps * 32/10 docs.
# WTT converges fast (~10K steps) since CSE waves already encode words.
texts = load_training_data(max_docs=250_000)
split_idx = int(len(texts) * 0.9)
train_texts = texts[:split_idx]
eval_texts = texts[split_idx:]
print(f"  Train: {len(train_texts):,} docs, Eval: {len(eval_texts):,} docs")
chunks_est = len(train_texts) * 10
steps_est = chunks_est // 32
print(f"  Estimated chunks: ~{chunks_est:,} → ~{steps_est:,} steps (single pass)")

# Create trainer
trainer = Phase9Trainer(
    flux_model=model,
    wave_chunker=chunker,
    wave_generator=generator,
    wave_to_text=wtt,
    lr=3e-4,
    grad_accum=4,
    log=log,
)

log.cell_end("Cell 5 — Build Phase 9 Modules", "PASS")

---
## Cell 5b: Pre-Training Validation & Fresh Module Init

**Why this cell exists:** Kaggle kernels keep Python objects in memory across cell re-runs. Even though Cell 1 pulls fresh code and Cell 5 builds modules, re-running Cell 6 alone reuses the *already-trained* `wtt`/`chunker`/`generator` objects. This cell forces a complete teardown and rebuild so training always starts from scratch with the latest code.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5b — Pre-Training Validation & Fresh Module Init
# ═══════════════════════════════════════════════════════════════
import os, sys, gc, importlib, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/FLUX")

# ─────────────────────────────────────────────
# 1. Validate working directory
# ─────────────────────────────────────────────
cwd = Path.cwd()
if cwd != WORK_DIR:
    print(f"  ⚠ Wrong cwd: {cwd}")
    os.chdir(str(WORK_DIR))
    print(f"  ✓ Changed to: {WORK_DIR}")
else:
    print(f"  ✓ Working directory: {cwd}")

assert (WORK_DIR / 'phases' / 'phase9' / 'wave_to_text.py').exists(), \
    "ERROR: wave_to_text.py not found — did Cell 1 run?"

# ─────────────────────────────────────────────
# 2. Verify the EOS fix is present in code on disk
# ─────────────────────────────────────────────
_wtt_src = (WORK_DIR / 'phases' / 'phase9' / 'wave_to_text.py').read_text()
assert 'max_len + 1' in _wtt_src, \
    "ERROR: wave_to_text.py missing EOS fix (max_len + 1) — pull latest from GitHub!"
print(f"  ✓ wave_to_text.py has EOS fix (max_len + 1)")

# ─────────────────────────────────────────────
# 3. Destroy existing model objects & free GPU
# ─────────────────────────────────────────────
_destroyed = []
for _name in ['wtt', 'chunker', 'generator', 'trainer']:
    if _name in dir():
        _destroyed.append(_name)
    # Also check globals (notebook scope)
    if _name in globals():
        obj = globals()[_name]
        if hasattr(obj, 'cpu'):
            obj.cpu()
        del globals()[_name]
        _destroyed.append(_name)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

if _destroyed:
    print(f"  ✓ Destroyed cached objects: {list(set(_destroyed))}")
else:
    print(f"  ℹ No cached objects found")

# ─────────────────────────────────────────────
# 4. Flush ALL phase9-related modules from cache
#    so Python re-reads the .py files from disk
# ─────────────────────────────────────────────
_phase9_prefixes = (
    'wave_chunker', 'wave_generator', 'wave_to_text', 'wave_sampler',
    'wave_decoder', 'train_wave_gen',
)
_flushed = [m for m in list(sys.modules.keys()) if any(m.startswith(p) for p in _phase9_prefixes)]
for m in _flushed:
    del sys.modules[m]
if _flushed:
    print(f"  ✓ Flushed {len(_flushed)} cached modules: {_flushed}")
else:
    print(f"  ℹ No phase9 modules in cache")

# ─────────────────────────────────────────────
# 5. Fresh import & rebuild ALL Phase 9 modules
# ─────────────────────────────────────────────
from train_wave_gen import build_phase9_modules, Phase9Trainer, load_training_data, generate_text, PHASE9_CONFIG
from wave_sampler import ThermodynamicWaveSampler

chunker, generator, wtt = build_phase9_modules(device=DEVICE)
print(f"  ✓ Built fresh modules: WaveChunker, WaveGenerator, WaveToText")
print(f"    WTT params: {sum(p.numel() for p in wtt.parameters()):,}")

# ─────────────────────────────────────────────
# 6. Sanity check: untrained WTT should have HIGH loss
# ─────────────────────────────────────────────
with torch.no_grad():
    _test_wave = model.cse.encode("hello")
    _test_seq = _test_wave.full.to(DEVICE)
    _pairs = chunker.chunk_with_bytes(_test_seq, b"hello")
    if _pairs:
        _chunk_w, _chunk_b = _pairs[0]
        _byte_t = torch.tensor(list(_chunk_b), dtype=torch.long, device=DEVICE)
        _logits = wtt(_chunk_w, _byte_t)
        _targets = torch.tensor(list(_chunk_b) + [wtt.vocab_size], dtype=torch.long, device=DEVICE)
        _loss = torch.nn.functional.cross_entropy(_logits, _targets)
        print(f"\n  ✓ Fresh WTT sanity loss: {_loss.item():.4f}")
        if _loss.item() < 0.5:
            print(f"  ⚠ WARNING: Loss suspiciously low ({_loss.item():.4f}) — weights may still be cached!")
            print(f"    Expected ~5.5 for untrained model (log(258) ≈ 5.55)")
        else:
            print(f"  ✓ Loss is high as expected for untrained model (good!)")

# ─────────────────────────────────────────────
# 7. Rebuild trainer with fresh modules
# ─────────────────────────────────────────────
trainer = Phase9Trainer(
    flux_model=model,
    wave_chunker=chunker,
    wave_generator=generator,
    wave_to_text=wtt,
    lr=3e-4,
    grad_accum=4,
    log=log,
)
print(f"  ✓ Trainer rebuilt with fresh modules")
print(f"\n{'═' * 50}")
print(f"  Ready for Cell 6 — training will start from scratch")
print(f"{'═' * 50}")

---
## Cell 6: Stage 1 — WaveToText Pre-Training (60K steps, ~15-20 min)

Train the WaveToText decoder to convert CSE waves back to readable text. This is the "last-mile" component that makes wave-level generation produce words. 60K steps achieves ~93% word accuracy.

In [ ]:
log.cell_start("Cell 6 — Stage 1: WaveToText Pre-Training")

wtt_result = trainer.train_wave_to_text(
    train_texts,
    max_steps=60000,
    batch_size=32,
    log_interval=2000,
)

log.metric("wtt_steps", str(wtt_result.total_steps))
log.metric("wtt_word_accuracy", f"{wtt_result.word_accuracy:.1%}")
log.metric("wtt_final_loss", f"{wtt_result.final_loss:.4f}")
log.metric("wtt_time", f"{wtt_result.total_time_seconds:.1f}s")

log.cell_end("Cell 6 — Stage 1: WaveToText Pre-Training", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6b — DEBUG: WaveToText Decode Diagnostic
# ═══════════════════════════════════════════════════════════════
import torch

wtt.eval()
chunker.eval()

# Pick a few short test words
test_texts = [
    "Hello world",
    "The cat sat on the mat",
    "artificial intelligence",
    "energy",
    "physics",
]

print("═" * 70)
print("  WaveToText Decode Diagnostic")
print("═" * 70)

for text in test_texts:
    with torch.no_grad():
        wave = model.cse.encode(text)
        wave_seq = wave.full.to(DEVICE)
        text_bytes = text.encode('utf-8', errors='replace')

        pairs = chunker.chunk_with_bytes(wave_seq, text_bytes)

    print(f"\n  Text: \"{text}\"")
    print(f"  Chunks: {len(pairs)}")

    for i, (chunk_wave, chunk_bytes) in enumerate(pairs[:5]):
        expected = chunk_bytes.decode('utf-8', errors='replace')

        # Test 1: Teacher-forced logits — does the model "know" the answer?
        with torch.no_grad():
            byte_tensor = torch.tensor(list(chunk_bytes), dtype=torch.long, device=DEVICE)
            logits = wtt(chunk_wave, byte_tensor)  # [chunk_len+1, 257]
            preds = logits.argmax(dim=-1)
            # Targets: bytes + EOS
            target_with_eos = list(chunk_bytes) + [wtt.vocab_size]
            correct_bytes = sum(1 for p, t in zip(preds.tolist(), target_with_eos) if p == t)
            tf_acc = correct_bytes / len(target_with_eos)

        # Test 2: Greedy decode (temperature=0.01 ≈ argmax)
        with torch.no_grad():
            decoded_greedy = wtt.decode(chunk_wave, temperature=0.01)
            greedy_text = decoded_greedy.decode('utf-8', errors='replace')

        # Test 3: Normal decode (temperature=0.5 — what eval uses)
        with torch.no_grad():
            decoded_t05 = wtt.decode(chunk_wave, temperature=0.5)
            t05_text = decoded_t05.decode('utf-8', errors='replace')

        # Test 4: Check raw logits at first step
        with torch.no_grad():
            hidden = wtt.wave_to_hidden(chunk_wave).unsqueeze(0).unsqueeze(0)
            bos_token = torch.full((1,), wtt.BOS, dtype=torch.long, device=DEVICE)
            embedded = wtt.byte_embed(bos_token).unsqueeze(0)
            output, _ = wtt.gru(embedded, hidden)
            first_logits = wtt.output_proj(output.squeeze(0).squeeze(0))
            top5 = torch.topk(first_logits, 5)
            top5_bytes = [(idx.item(), f"'{chr(idx.item())}'" if idx.item() < 128 else f"0x{idx.item():02x}")
                          for idx in top5.indices]
            expected_first = chunk_bytes[0] if chunk_bytes else -1

        exact_match = "✓" if decoded_greedy == chunk_bytes else "✗"

        print(f"\n    Chunk {i}: expected=\"{expected}\" ({len(chunk_bytes)} bytes)")
        print(f"      Teacher-forced byte acc: {tf_acc:.0%} ({correct_bytes}/{len(target_with_eos)})")
        print(f"      Greedy decode:   \"{greedy_text}\" {exact_match}")
        print(f"      Temp=0.5 decode: \"{t05_text}\"")
        print(f"      First byte — expected: {expected_first} ('{chr(expected_first)}')")
        print(f"      First byte — top5 logits: {top5_bytes}")
        print(f"      First byte — top5 probs:  {torch.softmax(first_logits, -1)[top5.indices].tolist()}")

wtt.train()
chunker.train()

print("\n" + "═" * 70)
print("  If teacher-forced acc is high but greedy fails → exposure bias")
print("  If teacher-forced acc is low → training didn't work (check loss)")
print("  If greedy works but temp=0.5 fails → temperature too high")
print("  If first-byte top5 is wrong → wave→hidden projection is broken")
print("═" * 70)

---
## Cell 7: Stage 2 — WaveGenerator Training

Train the WaveGenerator to predict the next wave given previous waves + field context. Pre-computes all frozen pipeline outputs (CSE, field, CGN, chunker) upfront, then trains purely on GPU — eliminating the CPU bottleneck.

---
## Cell 7b: WaveGenerator Fresh Init (keeps trained WTT)

Same idea as Cell 5b but for Stage 2. Destroys the cached WaveGenerator so training starts from random weights. **Keeps the trained WTT and WaveChunker** from Stage 1 — only the generator is rebuilt.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7b — WaveGenerator Fresh Init (keeps trained WTT)
# ═══════════════════════════════════════════════════════════════
import sys, gc, importlib, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/FLUX")

# ─────────────────────────────────────────────
# 1. Validate working directory
# ─────────────────────────────────────────────
import os
cwd = Path.cwd()
if cwd != WORK_DIR:
    os.chdir(str(WORK_DIR))
    print(f"  ✓ Changed to: {WORK_DIR}")
else:
    print(f"  ✓ Working directory: {cwd}")

# ─────────────────────────────────────────────
# 2. Verify trained WTT is in memory
# ─────────────────────────────────────────────
_have_wtt = 'wtt' in dir() or 'wtt' in globals()
_have_chunker = 'chunker' in dir() or 'chunker' in globals()
_have_model = 'model' in dir() or 'model' in globals()

if not _have_wtt:
    raise RuntimeError("No trained WTT found! Run Cells 5b + 6 first.")
if not _have_chunker:
    raise RuntimeError("No WaveChunker found! Run Cells 5b + 6 first.")
if not _have_model:
    raise RuntimeError("No FLUX model found! Run Cell 4 first.")

# Quick check: WTT should have LOW loss if trained
with torch.no_grad():
    _tw = model.cse.encode("hello")
    _ts = _tw.full.to(DEVICE)
    _tp = chunker.chunk_with_bytes(_ts, b"hello")
    if _tp:
        _cw, _cb = _tp[0]
        _bt = torch.tensor(list(_cb), dtype=torch.long, device=DEVICE)
        _lo = wtt(_cw, _bt)
        _tg = torch.tensor(list(_cb) + [wtt.vocab_size], dtype=torch.long, device=DEVICE)
        _wtt_loss = torch.nn.functional.cross_entropy(_lo, _tg).item()
        print(f"  ✓ Trained WTT loss: {_wtt_loss:.4f}")
        if _wtt_loss > 2.0:
            print(f"  ⚠ WARNING: WTT loss is high ({_wtt_loss:.4f}) — is it actually trained?")
        else:
            print(f"  ✓ WTT is trained (loss < 2.0)")

# ─────────────────────────────────────────────
# 3. Destroy ONLY the generator (keep WTT + chunker)
# ─────────────────────────────────────────────
if 'generator' in globals():
    old_gen = globals()['generator']
    if hasattr(old_gen, 'cpu'):
        old_gen.cpu()
    del globals()['generator']
    print(f"  ✓ Destroyed cached WaveGenerator")

if 'trainer' in globals():
    del globals()['trainer']
    print(f"  ✓ Destroyed cached trainer")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# ─────────────────────────────────────────────
# 4. Flush only wave_generator and train_wave_gen
#    modules so Python re-reads from disk
# ─────────────────────────────────────────────
_wg_mods = [m for m in list(sys.modules.keys())
            if m.startswith(('wave_generator', 'train_wave_gen'))]
for m in _wg_mods:
    del sys.modules[m]
if _wg_mods:
    print(f"  ✓ Flushed modules: {_wg_mods}")

# ─────────────────────────────────────────────
# 5. Build FRESH WaveGenerator only
# ─────────────────────────────────────────────
from train_wave_gen import PHASE9_CONFIG, Phase9Trainer
from wave_generator import WaveGenerator

generator = WaveGenerator(
    wave_dim=PHASE9_CONFIG['wave_dim'],
    field_features=PHASE9_CONFIG['field_features'],
    max_waves=PHASE9_CONFIG['max_waves'],
    k_neighbors=PHASE9_CONFIG['k_neighbors'],
    interference_radius=PHASE9_CONFIG['interference_radius'],
).to(DEVICE)

gen_params = sum(p.numel() for p in generator.parameters())
print(f"  ✓ Fresh WaveGenerator: {gen_params:,} params")

# ─────────────────────────────────────────────
# 6. Sanity check: untrained WG should have random
#    cosine similarity to targets (~0.0 on average)
# ─────────────────────────────────────────────
with torch.no_grad():
    _wave = model.cse.encode("The quick brown fox jumps over the lazy dog")
    _seq = _wave.full.to(DEVICE)
    _vec = _seq.mean(dim=0)
    _ff, _ss, _ll = model.field.query(_vec, k=4)
    _merged = _ff.mean(dim=0) + model.cgn(_ff.mean(dim=0))
    _chunks, _spans = chunker(_seq)

    if _chunks.shape[0] >= 2:
        _pred, _conf = generator(_merged, _chunks)
        _cos = torch.nn.functional.cosine_similarity(_pred, _chunks, dim=-1).mean().item()
        _mse = torch.nn.functional.mse_loss(_pred, _chunks).item()
        print(f"\n  ✓ Fresh WG sanity check:")
        print(f"    Cosine similarity: {_cos:.4f} (should be near 0.0)")
        print(f"    MSE loss: {_mse:.4f} (should be high)")
        if _cos > 0.5:
            print(f"  ⚠ WARNING: Cosine suspiciously high ({_cos:.4f}) — weights may be cached!")
        else:
            print(f"  ✓ Generator is fresh/untrained (good!)")

# ─────────────────────────────────────────────
# 7. Rebuild trainer with trained WTT + fresh WG
# ─────────────────────────────────────────────
trainer = Phase9Trainer(
    flux_model=model,
    wave_chunker=chunker,
    wave_generator=generator,
    wave_to_text=wtt,
    lr=3e-4,
    grad_accum=4,
    log=log,
)

wtt_params = sum(p.numel() for p in wtt.parameters())
print(f"\n  ✓ Trainer rebuilt:")
print(f"    WTT: {wtt_params:,} params (TRAINED, frozen in Stage 2)")
print(f"    WG:  {gen_params:,} params (FRESH, will be trained)")

print(f"\n{'═' * 50}")
print(f"  Ready for Cell 7 — WG training starts from scratch")
print(f"  WTT is preserved from Cell 6")
print(f"{'═' * 50}")

In [ ]:
log.cell_start("Cell 7 — Stage 2: WaveGenerator Training")

wg_result = trainer.train_wave_generator(
    train_texts,
    max_steps=5000,
    log_interval=50,
)

log.metric("wg_steps", str(wg_result.total_steps))
log.metric("wg_final_loss", f"{wg_result.final_loss:.4f}")
log.metric("wg_cosine_accuracy", f"{wg_result.wave_cosine_accuracy:.3f}")
log.metric("wg_time", f"{wg_result.total_time_seconds:.1f}s")

total_time = wtt_result.total_time_seconds + wg_result.total_time_seconds
print(f"\n  Total training time: {total_time:.1f}s ({total_time/3600:.1f}h)")

log.cell_end("Cell 7 — Stage 2: WaveGenerator Training", "PASS")

---
## Cell 8: Save Checkpoint + Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 8 — Save & Upload Checkpoint")

from flux_utils import upload_checkpoint_to_hf

# Quick valid word rate evaluation for checkpoint
prompts = [
    "The future of artificial intelligence",
    "In the beginning",
    "Scientists have discovered",
    "The relationship between energy and matter",
    "Modern technology relies on",
]
valid_words = 0
total_words = 0
for p in prompts:
    try:
        result = generate_text(p, model, chunker, generator, wtt, max_waves=15, temperature=0.8)
        continuation = result[len(p):].strip()
        for w in continuation.split():
            clean = w.strip('.,;:!?"\'-()[]{}').lower()
            if clean.isalpha() and len(clean) >= 2:
                total_words += 1
                if len(clean) <= 15:
                    valid_words += 1
    except Exception:
        pass
valid_rate = valid_words / max(total_words, 1)
print(f"  Valid word rate: {valid_rate:.1%}")

# Save checkpoint
ckpt_path = trainer.save_phase9_checkpoint(wtt_result, wg_result, valid_rate)

# Upload to HuggingFace Hub
if HF_TOKEN:
    try:
        upload_checkpoint_to_hf(phase=9, hf_token=HF_TOKEN)
        log.success("Checkpoint uploaded to HuggingFace Hub")
    except Exception as e:
        log.warning(f"HF upload failed: {e}")
else:
    log.warning("No HF token — skipping upload")

log.cell_end("Cell 8 — Save & Upload Checkpoint", "PASS")

---
## Cells 9–11: Tests

| Test | What it checks | Threshold |
|------|---------------|-----------|
| Test 1 | Generated wave distributions match training distribution | > 0.5 |
| Test 2 | WaveToText reconstructs words accurately | > 0.8 |
| Test 3 | Full pipeline produces valid English words | > 0.5 |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9 — Test 1: Wave Distribution Match
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 9 — Test 1: Wave Distribution Match")

import test_phase9_test1
test1_passed = test_phase9_test1.main()
status1 = "PASS" if test1_passed else "FAIL"

log.cell_end("Cell 9 — Test 1: Wave Distribution Match", status1)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 10 — Test 2: Word Reconstruction Accuracy
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 10 — Test 2: Word Reconstruction")

import test_phase9_test2
test2_passed = test_phase9_test2.main()
status2 = "PASS" if test2_passed else "FAIL"

log.cell_end("Cell 10 — Test 2: Word Reconstruction", status2)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 11 — Test 3: Full Pipeline English Words
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 11 — Test 3: Full Pipeline English Words")

import test_phase9_test3
test3_passed = test_phase9_test3.main()
status3 = "PASS" if test3_passed else "FAIL"

log.cell_end("Cell 11 — Test 3: Full Pipeline English Words", status3)

---
## Cells 12–14: Demos

| Demo | Description |
|------|------------|
| Demo 1 | Wave-level generation with multiple prompts |
| Demo 2 | Wave sequence visualization (saved as PNG) |
| Demo 3 | Thermodynamic sampling energy trace (saved as PNG) |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 12 — Demo 1: Wave-Level Generation
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 12 — Demo 1: Wave-Level Generation")

import demo_phase9_demo1
demo_phase9_demo1.main()

log.cell_end("Cell 12 — Demo 1: Wave-Level Generation", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 13 — Demo 2: Wave Sequence Visualization
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 13 — Demo 2: Wave Sequence Visualization")

import demo_phase9_demo2
demo_phase9_demo2.main()

# Display the saved figure in notebook
from IPython.display import Image, display
fig_path = Path("phases/phase9/wave_sequence.png")
if fig_path.exists():
    display(Image(filename=str(fig_path), width=900))

log.cell_end("Cell 13 — Demo 2: Wave Sequence Visualization", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 14 — Demo 3: Thermodynamic Sampling Trace
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 14 — Demo 3: Thermodynamic Sampling Trace")

import demo_phase9_demo3
demo_phase9_demo3.main()

# Display the saved figure in notebook
fig_path = Path("phases/phase9/thermo_wave_sampling.png")
if fig_path.exists():
    display(Image(filename=str(fig_path), width=900))

log.cell_end("Cell 14 — Demo 3: Thermodynamic Sampling Trace", "PASS")

---
## Cell 15: Interactive Exploration

In [ ]:
log.cell_start("Cell 15 — Interactive Exploration")

# Try your own prompts!
test_prompts = [
    "Hello, my name is",
    "The meaning of life is",
    "In a galaxy far far away",
    "Artificial intelligence will",
    "The ocean is vast and",
]

print("═" * 60)
print("  Interactive Wave Generation")
print("═" * 60)

for prompt in test_prompts:
    try:
        result = generate_text(
            prompt, model, chunker, generator, wtt,
            max_waves=15, temperature=0.8, use_sampler=True,
        )
        print(f"\n  Prompt: \"{prompt}\"")
        print(f"  Output: {result[:200]}")
    except Exception as e:
        print(f"\n  Prompt: \"{prompt}\"")
        print(f"  Error: {e}")

log.cell_end("Cell 15 — Interactive Exploration", "PASS")

---
## Cell 16: Results Summary

In [ ]:
log.cell_start("Cell 16 — Results Summary")

results = PhaseResults(phase=9, component_name="Wave-Level Generation")

# Training metrics
results.add_metric("WTT training steps", wtt_result.total_steps)
results.add_metric("WTT word accuracy", f"{wtt_result.word_accuracy:.1%}")
results.add_metric("WTT final loss", f"{wtt_result.final_loss:.4f}")
results.add_metric("WG training steps", wg_result.total_steps)
results.add_metric("WG final loss", f"{wg_result.final_loss:.4f}")
results.add_metric("WG cosine accuracy", f"{wg_result.wave_cosine_accuracy:.3f}")
results.add_metric("Valid word rate", f"{valid_rate:.1%}")
results.add_metric("Total training time", f"{total_time:.1f}s ({total_time/3600:.1f}h)")

# Test results
results.add_test("Wave Distribution Match", passed=test1_passed, score=0.0, threshold=0.5)
results.add_test("Word Reconstruction", passed=test2_passed, score=0.0, threshold=0.8)
results.add_test("Full Pipeline English", passed=test3_passed, score=0.0, threshold=0.5)

# Demo results
results.add_demo("Wave-Level Generation", ran=True, quality="See output above")
results.add_demo("Wave Sequence Visualization", ran=True, quality="See figure above")
results.add_demo("Thermodynamic Sampling Trace", ran=True, quality="See figure above")

md_output = results.save()

# Display formatted results
from IPython.display import Markdown, display
results_path = Path('phases/phase9/RESULTS_PHASE_9.md')
if results_path.exists():
    display(Markdown(results_path.read_text()))
else:
    print(md_output)

log.cell_end("Cell 16 — Results Summary", "PASS")

---
## Cell 17: View Full Log

In [ ]:
print("=" * 60)
print("  FULL PHASE 9 LOG")
print("=" * 60)
print(log.get_log_contents())

---
## Cell 18: Final Upload (Logs → HF Hub + GitHub)

In [ ]:
log.cell_start("Cell 18 — Final Upload")

WORK_DIR = Path("/kaggle/working/FLUX")

# Upload logs to HuggingFace
if HF_TOKEN:
    try:
        upload_logs_to_hf(phase=9, hf_token=HF_TOKEN)
        log.success("Logs uploaded to HuggingFace Hub")
    except Exception as e:
        log.warning(f"Log upload failed: {e}")

# Git commit and push
try:
    git_commit_and_push(
        files=[
            'logs/phase9.log',
            'phases/phase9/RESULTS_PHASE_9.md',
            'results/RESULTS_PHASE_9.md',
        ],
        message='Phase 9: Wave-Level Generation complete [auto-commit from Kaggle]',
        repo_dir=str(WORK_DIR),
    )
    log.success("Committed and pushed to GitHub")
except Exception as e:
    log.warning(f"Git push failed: {e}")

log.cell_end("Cell 18 — Final Upload", "PASS")

print("\n" + "=" * 60)
print("✓ PHASE 9 COMPLETE")
print("=" * 60)
print(f"  Checkpoint: https://huggingface.co/UnseenGAP/FLUX")
print(f"  Logs:       https://huggingface.co/UnseenGAP/FLUX (logs/)")
print(f"  Code:       https://github.com/Unseengap/FLUX")
print("=" * 60)

---
## Cell 19: Save Artifacts to Kaggle Output

In [ ]:
log.cell_start("Cell 19 — Save Kaggle Artifacts")

import shutil

WORK_DIR = Path("/kaggle/working/FLUX")
output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# ── Checkpoint ──
ckpt_src = WORK_DIR / 'checkpoints' / 'phase9.phase.pt'
if ckpt_src.exists():
    shutil.copy2(str(ckpt_src), str(output_dir / ckpt_src.name))
    print(f"  ✓ Checkpoint: {ckpt_src.name} ({ckpt_src.stat().st_size/1e6:.1f} MB)")

# ── Results and logs ──
for fpath in [
    WORK_DIR / 'phases' / 'phase9' / 'RESULTS_PHASE_9.md',
    WORK_DIR / 'results' / 'RESULTS_PHASE_9.md',
    WORK_DIR / 'logs' / 'phase9.log',
]:
    if fpath.exists():
        shutil.copy2(str(fpath), str(output_dir / fpath.name))
        print(f"  ✓ {fpath.name}")

# ── Figures ──
for fig_name in ['wave_sequence.png', 'thermo_wave_sampling.png']:
    src = WORK_DIR / 'phases' / 'phase9' / fig_name
    if src.exists():
        shutil.copy2(str(src), str(output_dir / src.name))
        print(f"  ✓ {fig_name}")

# ── List all saved artifacts ──
print("\n  Saved artifacts:")
for f in sorted(output_dir.iterdir()):
    size = f.stat().st_size
    unit = 'MB' if size > 1e6 else 'KB'
    val = size / 1e6 if size > 1e6 else size / 1e3
    print(f"    {f.name:40s} {val:8.1f} {unit}")

log.cell_end("Cell 19 — Save Kaggle Artifacts", "PASS")

print("\n" + "=" * 60)
print("ALL DONE — Phase 9: Wave-Level Generation")
print("=" * 60)

---
## Acceptance Criteria

| Criterion | Target | Method |
|-----------|--------|--------|
| WaveToText word accuracy | ≥ 80% | Stage 1 training metric |
| WaveGenerator cosine accuracy | ≥ 0.5 | Stage 2 training metric |
| Valid English word rate | ≥ 50% | Full pipeline evaluation |
| Wave distribution match | ≥ 0.5 | Test 1 |
| All 3 tests pass | 3/3 | Cells 9–11 |

### Key Design Decisions

- **Wave-level, not byte-level:** Generates semantic wave chunks (~5-15 per sentence) instead of individual bytes (~100+ per sentence)
- **Universal core:** WaveGenerator operates on wave vectors — same mechanism works for any modality
- **Two-stage training:** WaveToText first (supervised), then WaveGenerator (wave prediction)
- **Thermodynamic sampling:** Energy-based wave selection replaces simple temperature scaling
- **Builds on Phase 7:** Uses frozen CSE, field, GR, TL, CGN, memory from prior phases
- **Peer to Phase 8:** Not a continuation — an alternative generation paradigm

### Full Pipeline

```
text → CSE(encode) → [seq, 432] wave
     → WaveChunker → list of wave chunks
     → Field(query) → [k, 512] context features
     → WaveGenerator → next wave prediction (wave-by-wave)
     → ThermodynamicWaveSampler → energy-based selection
     → WaveToText → readable text output
```